In [1]:
import sys
import os
import time

# 让 import config 能找到项目根目录
sys.path.insert(0, os.path.abspath(".."))

from config import chat, get_llm_config, print_config


def header(title: str):
    cfg = get_llm_config()
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"  后端: {cfg['backend']} | 模型: {cfg['model']}")
    print(f"{'='*60}")


def run_comparison(prompt: str, param_name: str, values: list, fixed_options: dict = None, runs: int = 2):
    """通用对比函数：对同一 prompt，变化某个参数，每个值跑 runs 次。"""
    fixed = fixed_options or {}
    for val in values:
        print(f"\n  {param_name}={val}:")
        opts = {**fixed, param_name: val}
        for r in range(runs):
            result = chat([{"role": "user", "content": prompt}], **opts)
            # 只取第一行，避免输出太长
            first_line = result.strip().split("\n")[0]
            print(f"    第{r+1}次: {first_line}")


def is_ollama() -> bool:
    """检查当前是否使用 Ollama 后端。"""
    return get_llm_config()["backend"] == "ollama"

In [2]:
# ============================================================
# 实验 1: Temperature
# ============================================================
def experiment_temperature():
    """
    Temperature 控制输出的随机性。
    - 低温 (0.1): 几乎确定性输出，两次结果基本相同
    - 中温 (0.7): 自然多样，适合对话
    - 高温 (1.5): 高度随机，可能出现意想不到的表达

    Ollama 和 Groq 都支持此参数。
    """
    header("实验 1: Temperature（温度）")

    prompt = "/no_think 用一句话描述春天"
    print(f"  Prompt: {prompt}")
    print(f"  观察: 温度越低，两次输出越相似")

    run_comparison(prompt, "temperature", [0.1, 0.5, 0.7, 1.0, 1.5],
                   fixed_options={"num_ctx": 4096})

    # 补充：temperature=0 的确定性
    print(f"\n  --- temperature=0（完全确定性，3次应完全相同）---")
    for r in range(3):
        result = chat([{"role": "user", "content": prompt}], temperature=0, num_ctx=4096)
        first_line = result.strip().split("\n")[0]
        print(f"    第{r+1}次: {first_line}")

In [3]:
experiment_temperature()


  实验 1: Temperature（温度）
  后端: groq | 模型: qwen/qwen3-32b
  Prompt: /no_think 用一句话描述春天
  观察: 温度越低，两次输出越相似

  temperature=0.1:
    第1次: 春天是万物复苏、花开鸟鸣、大地披上新绿的温暖季节。
    第2次: 春天是万物复苏、花开鸟鸣、大地披上新绿的季节。

  temperature=0.5:
    第1次: 春天是万物复苏、花开鸟鸣、温暖而充满生机的季节。
    第2次: 春天是万物复苏、花开鸟鸣的季节。

  temperature=0.7:
    第1次: 春天是万物复苏、花开鸟鸣的季节。
    第2次: 春天是万物复苏、花开鸟鸣、大地披上新绿的温暖季节。

  temperature=1.0:
    第1次: 春天是万物复苏、百花齐放的季节。
    第2次: 春天是万物复苏、花开鸟鸣的生机盎然的季节。

  temperature=1.5:
    第1次: 春天是万物复苏、鲜花盛开、生机勃勃的季节。
    第2次: 春天是万物复苏、生机勃勃的季节。

  --- temperature=0（完全确定性，3次应完全相同）---
    第1次: 春天是万物复苏、花开鸟鸣的季节。
    第2次: 春天是万物复苏、花开鸟鸣的季节。
    第3次: 春天是万物复苏、花开鸟鸣的季节。


In [4]:
# ============================================================
# 实验 2: top_p (Nucleus Sampling)
# ============================================================
def experiment_top_p():
    """
    top_p 截断候选 token 集合。Ollama 和 Groq 都支持。
    """
    header("实验 2: top_p（核采样）")

    prompt = "用一句话描述春天"
    print(f"  Prompt: {prompt}")
    print(f"  固定: temperature=0.8")
    print(f"  观察: top_p 越小，输出越保守/重复")

    run_comparison(prompt, "top_p", [0.1, 0.3, 0.5, 0.9, 1.0],
                   fixed_options={"temperature": 0.8, "num_ctx": 4096})

    print(f"\n  --- 换一个更有创意空间的 prompt ---")
    creative = "用一个比喻描述人工智能"
    print(f"  Prompt: {creative}")

    run_comparison(creative, "top_p", [0.3, 0.9],
                   fixed_options={"temperature": 0.8, "num_ctx": 4096}, runs=3)


In [6]:
experiment_top_p()


  实验 2: top_p（核采样）
  后端: groq | 模型: qwen/qwen3-32b
  Prompt: 用一句话描述春天
  固定: temperature=0.8
  观察: top_p 越小，输出越保守/重复

  top_p=0.1:
    第1次: "春天是大地苏醒的画卷，暖阳轻吻新绿，万物在芬芳中舒展生命的诗意。"
    第2次: "春天是大地苏醒的画卷，暖阳轻吻新绿，万物在芬芳中舒展生命的诗意。"

  top_p=0.3:
    第1次: 春天是万物复苏的季节，微风轻拂，百花绽放，大地披上绿装，生机盎然。
    第2次: 春天是万物复苏的季节，暖风轻拂，花开遍野，大地披上绿意盎然的新装。

  top_p=0.5:
    第1次: 春天是苏醒的大地在阳光下舒展身姿，万物萌发，百花争艳，连空气都浸润着生命初绽的芬芳。
    第2次: 春天是万物复苏的季节，大地披上绿装，花朵绽放，鸟儿欢唱，温暖的阳光唤醒沉睡的生机。

  top_p=0.9:
    第1次: "春风裹着桃李的芬芳，掠过苏醒的原野，万物在细雨中舒展成一首翠绿的诗。"
    第2次: 春天是万物复苏的季节，温暖的阳光唤醒沉睡的大地，花儿绽放，绿意盎然，生机勃勃。

  top_p=1.0:
    第1次: 春是苏醒的大地在枝头颤动，细雨揉开新绿，风里漾着桃李的芬芳，时光忽然就染上了花的颜色。
    第2次: 春天是万物苏醒的季节，冰雪消融，草木萌发，花开满枝，微风拂过带来泥土的芬芳和生命的律动。

  --- 换一个更有创意空间的 prompt ---
  Prompt: 用一个比喻描述人工智能

  top_p=0.3:
    第1次: 人工智能如同一位交响乐团的指挥家，手持无形的指挥棒，在数据的乐谱上编织旋律。它并非创造音乐本身，而是通过理解每个乐器（算法）的特性，协调音符（数据）的节奏与力度，将杂乱的原始音符转化为和谐的乐章。有时它会根据观众的掌声（反馈）调整指挥动作，甚至在乐谱未写完时即兴发挥（机器学习），但始终遵循作曲家（人类目标）的意图。这场演出没有固定剧本，却在规则与创新的张力中，不断演绎出新的可能。
    第2次: **人工智能如同一位技艺高超的调酒师**，在浩瀚的配方书（数据）中寻找灵感，通过反复品尝（训练）调整配方比例（参数），

In [ ]:
# ============================================================
# 实验 3: top_k（仅 Ollama）
# ============================================================
def experiment_top_k():
    """top_k 限制只从概率最高的 K 个 token 中采样。Groq 不支持。"""
    if not is_ollama():
        print("\n  ⚠ top_k 参数仅 Ollama 支持，Groq 不支持此参数，跳过。")
        return

    header("实验 3: top_k")

    prompt = "写一句关于月亮的诗句"
    print(f"  Prompt: {prompt}")
    print(f"  固定: temperature=0.8, num_ctx=4096")
    print(f"  观察: top_k 越小，输出越单调")

    run_comparison(prompt, "top_k", [1, 5, 10, 40, 100],
                   fixed_options={"temperature": 0.8, "num_ctx": 4096})


In [8]:
experiment_top_k()


  ⚠ top_k 参数仅 Ollama 支持，Groq 不支持此参数，跳过。


In [9]:
# ============================================================
# 实验 4: seed（可复现性）
# ============================================================
def experiment_seed():
    """seed 固定随机数生成器。Ollama 和 Groq 都支持。"""
    header("实验 4: seed（随机种子）")

    prompt = "用一句话描述春天"
    print(f"  Prompt: {prompt}")
    print(f"  固定: temperature=0.8")

    print(f"\n  --- 相同 seed=42，3次调用（应完全相同）---")
    for r in range(3):
        result = chat([{"role": "user", "content": prompt}],
                      temperature=0.8, seed=42, num_ctx=4096)
        first_line = result.strip().split("\n")[0]
        print(f"    第{r+1}次: {first_line}")

    print(f"\n  --- 不同 seed（应各不相同）---")
    for seed in [42, 123, 456, 999]:
        result = chat([{"role": "user", "content": prompt}],
                      temperature=0.8, seed=seed, num_ctx=4096)
        first_line = result.strip().split("\n")[0]
        print(f"    seed={seed}: {first_line}")

    print(f"\n  --- seed 在 temperature=0.1 下（差异很小）---")
    for seed in [42, 123]:
        result = chat([{"role": "user", "content": prompt}],
                      temperature=0.1, seed=seed, num_ctx=4096)
        first_line = result.strip().split("\n")[0]
        print(f"    seed={seed}: {first_line}")

In [10]:
experiment_seed()


  实验 4: seed（随机种子）
  后端: groq | 模型: qwen/qwen3-32b
  Prompt: 用一句话描述春天
  固定: temperature=0.8

  --- 相同 seed=42，3次调用（应完全相同）---
    第1次: "春天是阳光亲吻冻土时，万物睁开眼睛，风里漾着花香的序曲。"
    第2次: 春天是阳光轻吻大地的温柔诗篇，万物在细雨中舒展眉宇，花影摇曳成斑斓的序章。
    第3次: 春天是阳光轻吻大地的温柔诗篇，万物在细雨中舒展眉宇，花影摇曳成斑斓的序章。

  --- 不同 seed（应各不相同）---
    seed=42: **春天是阳光轻吻大地的刹那，万物在细雨中舒展眉宇，绽放出第一抹青涩的悸动。**  
    seed=123: "春风轻拂，万物复苏，桃花羞红了脸颊，燕子呢喃着春的序曲。"
    seed=456: “春天是时光在枝头绽放的微笑，冰雪吻醒泥土里沉睡的诗行，嫩绿与粉红的笔触蘸着微风，在蓝白交织的画布上写下重生的韵律。”
    seed=999: 春天是万物复苏的时节，微风轻拂，万物萌发，百花齐放，处处洋溢着生机与希望。

  --- seed 在 temperature=0.1 下（差异很小）---
    seed=42: 春天是万物苏醒的季节，温暖的阳光唤醒沉睡的大地，嫩绿的新芽破土而出，花儿绽放笑脸，空气中弥漫着生机与希望。
    seed=123: 春天是大地轻柔的呼吸，唤醒沉睡的花朵，让阳光在绿意中跳起轻快的舞步。
